# Lesson 17: Deploying Scalable Agents in Production

In this notebook, you will:
1. Build the **SaaS Subscription Billing & Customer Support Agent** with response caching, complexity routing (`gemini-2.5-flash-lite` vs `gemini-2.5-flash`), OpenTelemetry spans, and a `$50` HITL credit gate.
2. Run an automated `evaluation_gate` and post-deploy JSON smoke-test suite.

In [ ]:
!pip install -q -U google-genai opentelemetry-api opentelemetry-sdk python-dotenv

In [ ]:
response_cache = {}
stats = {"cache": 0, "slm": 0, "llm": 0}

def route_and_answer(query: str) -> str:
    q = query.strip().lower()
    if q in response_cache:
        stats["cache"] += 1
        return response_cache[q]
    if any(w in q for w in ["refund", "dispute", "cancel", "invoice"]):
        stats["llm"] += 1
        ans = "[gemini-2.5-flash] Checked billing policy & invoice records."
    else:
        stats["slm"] += 1
        ans = "[gemini-2.5-flash-lite] Subscription is active."
    response_cache[q] = ans
    return ans

for q in ["status?", "status?", "check my refund invoice", "status?", "cancel policy?"]:
    route_and_answer(q)
print("Cost Report:", stats)